# Home Credit Default Risk - Feature Engineering multi-tables

**Objectif** : enrichir les données de `application_train.csv` avec les informations des 6 autres fichiers CSV.

Pour chaque client (`SK_ID_CURR`), on agrège l'historique financier (min, max, mean, sum, var) et on joint tout en un seul DataFrame.

Ce notebook couvre :
1. Preprocessing de application_train/test
2. Bureau + bureau_balance
3. Previous applications
4. POS CASH balance
5. Installments payments
6. Credit card balance
7. Jointure finale et export

## 1. Imports

In [13]:
import numpy as np
import pandas as pd
import gc
import time
import warnings
from contextlib import contextmanager

import matplotlib.pyplot as plt
import seaborn as sns

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')

## 2. Fonctions utilitaires

In [14]:
@contextmanager
def timer(title):
    """Mesure le temps d'exécution d'un bloc."""
    t0 = time.time()
    yield
    print(f'{title} - fait en {time.time() - t0:.0f}s')


def one_hot_encoder(df, nan_as_category=True):
    """One-hot encoding des colonnes catégorielles."""
    original_columns = list(df.columns)
    categorical_columns = [col for col in df.columns if df[col].dtype == 'object']
    df = pd.get_dummies(df, columns=categorical_columns, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns

## 3. Chargement des données nettoyées (output de kernel1)

On part des fichiers déjà nettoyés par kernel1 :
- Anomalies traitées
- Encodage ordinal et one-hot déjà appliqués
- Colonnes à >50% de NaN supprimées
- Features métier déjà créées

Kernel2 se concentre uniquement sur la jointure avec les 6 autres tables.

In [15]:
df      = pd.read_csv('data/app_train_clean.csv')
test_df = pd.read_csv('data/app_test_clean.csv')

print(f'Train nettoyé : {df.shape}')
print(f'Test  nettoyé : {test_df.shape}')
print(f'Doublons train : {df.duplicated().sum()} | Doublons test : {test_df.duplicated().sum()}')

# Concaténation pour traitement uniforme des jointures
df = pd.concat([df, test_df], ignore_index=True)
print(f'Shape après concat : {df.shape}')

del test_df
gc.collect()

Train nettoyé : (307511, 206)
Test  nettoyé : (48744, 205)
Doublons train : 0 | Doublons test : 0
Shape après concat : (356255, 206)


1529

## 4. Bureau + Bureau Balance

Historique des crédits du client dans d'autres institutions financières. On distingue les crédits **actifs** et **clos**.

In [16]:
def bureau_and_balance(num_rows=None, nan_as_category=True):
    bureau = pd.read_csv('data/bureau.csv',         nrows=num_rows)
    bb     = pd.read_csv('data/bureau_balance.csv', nrows=num_rows)

    bb,     bb_cat     = one_hot_encoder(bb,     nan_as_category)
    bureau, bureau_cat = one_hot_encoder(bureau, nan_as_category)

    # Agrégation bureau_balance par crédit
    bb_agg = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_agg[col] = ['mean']
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(bb_agg)
    bb_agg.columns = pd.Index([e[0] + '_' + e[1].upper() for e in bb_agg.columns.tolist()])
    bureau = bureau.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau.drop(['SK_ID_BUREAU'], axis=1, inplace=True)
    del bb, bb_agg
    gc.collect()

    # Agrégations numériques
    num_agg = {
        'DAYS_CREDIT':             ['min', 'max', 'mean', 'var'],
        'DAYS_CREDIT_ENDDATE':     ['min', 'max', 'mean'],
        'DAYS_CREDIT_UPDATE':      ['mean'],
        'CREDIT_DAY_OVERDUE':      ['max', 'mean'],
        'AMT_CREDIT_MAX_OVERDUE':  ['mean'],
        'AMT_CREDIT_SUM':          ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT':     ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE':  ['mean'],
        'AMT_CREDIT_SUM_LIMIT':    ['mean', 'sum'],
        'AMT_ANNUITY':             ['max', 'mean'],
        'CNT_CREDIT_PROLONG':      ['sum'],
        'MONTHS_BALANCE_MIN':      ['min'],
        'MONTHS_BALANCE_MAX':      ['max'],
        'MONTHS_BALANCE_SIZE':     ['mean', 'sum'],
    }
    cat_agg = {cat: ['mean'] for cat in bureau_cat}
    for cat in bb_cat:
        cat_agg[cat + '_MEAN'] = ['mean']

    bureau_agg = bureau.groupby('SK_ID_CURR').agg({**num_agg, **cat_agg})
    bureau_agg.columns = pd.Index(['BURO_' + e[0] + '_' + e[1].upper() for e in bureau_agg.columns.tolist()])

    # Crédits actifs - recherche dynamique de la colonne (le nom exact dépend du one-hot encoding)
    active_col = next((c for c in bureau.columns if 'CREDIT_ACTIVE' in c and 'Active' in c and 'nan' not in c), None)
    if active_col:
        active     = bureau[bureau[active_col] == 1]
        active_agg = active.groupby('SK_ID_CURR').agg(num_agg)
        active_agg.columns = pd.Index(['ACTIVE_' + e[0] + '_' + e[1].upper() for e in active_agg.columns.tolist()])
        bureau_agg = bureau_agg.join(active_agg, how='left', on='SK_ID_CURR')
        del active, active_agg
    gc.collect()

    # Crédits clos - recherche dynamique de la colonne
    closed_col = next((c for c in bureau.columns if 'CREDIT_ACTIVE' in c and 'Closed' in c and 'nan' not in c), None)
    if closed_col:
        closed     = bureau[bureau[closed_col] == 1]
        closed_agg = closed.groupby('SK_ID_CURR').agg(num_agg)
        closed_agg.columns = pd.Index(['CLOSED_' + e[0] + '_' + e[1].upper() for e in closed_agg.columns.tolist()])
        bureau_agg = bureau_agg.join(closed_agg, how='left', on='SK_ID_CURR')
        del closed, closed_agg
    del bureau
    gc.collect()
    return bureau_agg


with timer('Bureau + bureau_balance'):
    n_before = len(df)
    bureau = bureau_and_balance()
    print('Shape bureau agrégé :', bureau.shape)
    df = df.join(bureau, how='left', on='SK_ID_CURR')
    assert len(df) == n_before, f'Perte de lignes ! {n_before} -> {len(df)}'
    print(f'Lignes stables apres join bureau : {len(df)}')
    del bureau
    gc.collect()

Shape bureau agrégé : (305811, 27)
Lignes stables apres join bureau : 356255
Bureau + bureau_balance - fait en 23s


## 5. Previous Applications

Anciennes demandes de prêt chez Home Credit. On distingue les demandes **acceptées** et **refusées**.

In [17]:
def previous_applications(num_rows=None, nan_as_category=True):
    prev = pd.read_csv('data/previous_application.csv', nrows=num_rows)
    prev, cat_cols = one_hot_encoder(prev, nan_as_category)

    for col in ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
                'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
        prev[col].replace(365243, np.nan, inplace=True)

    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']

    num_agg = {
        'AMT_ANNUITY':            ['min', 'max', 'mean'],
        'AMT_APPLICATION':        ['min', 'max', 'mean'],
        'AMT_CREDIT':             ['min', 'max', 'mean'],
        'APP_CREDIT_PERC':        ['min', 'max', 'mean', 'var'],
        'AMT_DOWN_PAYMENT':       ['min', 'max', 'mean'],
        'AMT_GOODS_PRICE':        ['min', 'max', 'mean'],
        'HOUR_APPR_PROCESS_START':['min', 'max', 'mean'],
        'RATE_DOWN_PAYMENT':      ['min', 'max', 'mean'],
        'DAYS_DECISION':          ['min', 'max', 'mean'],
        'CNT_PAYMENT':            ['mean', 'sum'],
    }
    cat_agg = {cat: ['mean'] for cat in cat_cols}

    prev_agg = prev.groupby('SK_ID_CURR').agg({**num_agg, **cat_agg})
    prev_agg.columns = pd.Index(['PREV_' + e[0] + '_' + e[1].upper() for e in prev_agg.columns.tolist()])

    approved_col = next((c for c in prev.columns if 'NAME_CONTRACT_STATUS' in c and 'Approved' in c), None)
    if approved_col:
        approved     = prev[prev[approved_col] == 1]
        approved_agg = approved.groupby('SK_ID_CURR').agg(num_agg)
        approved_agg.columns = pd.Index(['APPROVED_' + e[0] + '_' + e[1].upper() for e in approved_agg.columns.tolist()])
        prev_agg = prev_agg.join(approved_agg, how='left', on='SK_ID_CURR')
        del approved, approved_agg

    refused_col = next((c for c in prev.columns if 'NAME_CONTRACT_STATUS' in c and 'Refused' in c), None)
    if refused_col:
        refused     = prev[prev[refused_col] == 1]
        refused_agg = refused.groupby('SK_ID_CURR').agg(num_agg)
        refused_agg.columns = pd.Index(['REFUSED_' + e[0] + '_' + e[1].upper() for e in refused_agg.columns.tolist()])
        prev_agg = prev_agg.join(refused_agg, how='left', on='SK_ID_CURR')
        del refused, refused_agg

    del prev
    gc.collect()
    return prev_agg


with timer('Previous applications'):
    n_before = len(df)
    prev = previous_applications()
    print('Shape previous agrégé :', prev.shape)
    df = df.join(prev, how='left', on='SK_ID_CURR')
    assert len(df) == n_before, f'Perte de lignes ! {n_before} -> {len(df)}'
    print(f'Lignes stables apres join previous : {len(df)}')
    del prev
    gc.collect()

Shape previous agrégé : (338857, 30)
Lignes stables apres join previous : 356255
Previous applications - fait en 16s


## 6. POS CASH Balance

Historique mensuel des prêts POS (point de vente) et cash.

In [18]:
def pos_cash(num_rows=None, nan_as_category=True):
    pos = pd.read_csv('data/POS_CASH_balance.csv', nrows=num_rows)
    pos, cat_cols = one_hot_encoder(pos, nan_as_category)

    agg = {
        'MONTHS_BALANCE': ['max', 'mean', 'size'],
        'SK_DPD':         ['max', 'mean'],
        'SK_DPD_DEF':     ['max', 'mean'],
    }
    for cat in cat_cols:
        agg[cat] = ['mean']

    pos_agg = pos.groupby('SK_ID_CURR').agg(agg)
    pos_agg.columns = pd.Index(['POS_' + e[0] + '_' + e[1].upper() for e in pos_agg.columns.tolist()])
    pos_agg['POS_COUNT'] = pos.groupby('SK_ID_CURR').size()

    del pos
    gc.collect()
    return pos_agg


with timer('POS CASH balance'):
    n_before = len(df)
    pos = pos_cash()
    print('Shape POS agrégé :', pos.shape)
    df = df.join(pos, how='left', on='SK_ID_CURR')
    assert len(df) == n_before, f'Perte de lignes ! {n_before} -> {len(df)}'
    print(f'Lignes stables apres join POS : {len(df)}')
    del pos
    gc.collect()

Shape POS agrégé : (337252, 8)
Lignes stables apres join POS : 356255
POS CASH balance - fait en 18s


## 7. Installments Payments

Historique des remboursements. On calcule les retards (`DPD`) et avances (`DBD`) de paiement.

In [19]:
def installments_payments(num_rows=None, nan_as_category=True):
    ins = pd.read_csv('data/installments_payments.csv', nrows=num_rows)
    ins, cat_cols = one_hot_encoder(ins, nan_as_category)

    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    ins['DPD'] = (ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']).clip(lower=0)
    ins['DBD'] = (ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']).clip(lower=0)

    agg = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD':                    ['max', 'mean', 'sum'],
        'DBD':                    ['max', 'mean', 'sum'],
        'PAYMENT_PERC':           ['max', 'mean', 'sum', 'var'],
        'PAYMENT_DIFF':           ['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT':         ['max', 'mean', 'sum'],
        'AMT_PAYMENT':            ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT':     ['max', 'mean', 'sum'],
    }
    for cat in cat_cols:
        agg[cat] = ['mean']

    ins_agg = ins.groupby('SK_ID_CURR').agg(agg)
    ins_agg.columns = pd.Index(['INSTAL_' + e[0] + '_' + e[1].upper() for e in ins_agg.columns.tolist()])
    ins_agg['INSTAL_COUNT'] = ins.groupby('SK_ID_CURR').size()

    del ins
    gc.collect()
    return ins_agg


with timer('Installments payments'):
    n_before = len(df)
    ins = installments_payments()
    print('Shape installments agrégé :', ins.shape)
    df = df.join(ins, how='left', on='SK_ID_CURR')
    assert len(df) == n_before, f'Perte de lignes ! {n_before} -> {len(df)}'
    print(f'Lignes stables apres join installments : {len(df)}')
    del ins
    gc.collect()

Shape installments agrégé : (339587, 26)
Lignes stables apres join installments : 356255
Installments payments - fait en 32s


## 8. Credit Card Balance

Historique mensuel des cartes de crédit.

In [21]:
def credit_card_balance(num_rows=None, nan_as_category=True):
    cc = pd.read_csv('data/credit_card_balance.csv', nrows=num_rows)
    cc, cat_cols = one_hot_encoder(cc, nan_as_category)
    cc.drop(['SK_ID_PREV'], axis=1, inplace=True)

    # Agrégation uniquement sur les colonnes numériques
    num_cols = cc.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != 'SK_ID_CURR']

    cc_agg = cc.groupby('SK_ID_CURR')[num_cols].agg(['min', 'max', 'mean', 'sum', 'var'])
    cc_agg.columns = pd.Index(['CC_' + e[0] + '_' + e[1].upper() for e in cc_agg.columns.tolist()])
    cc_agg['CC_COUNT'] = cc.groupby('SK_ID_CURR').size()

    del cc
    gc.collect()
    return cc_agg


with timer('Credit card balance'):
    n_before = len(df)
    cc = credit_card_balance()
    print('Shape credit card agrégé :', cc.shape)
    df = df.join(cc, how='left', on='SK_ID_CURR')
    assert len(df) == n_before, f'Perte de lignes ! {n_before} -> {len(df)}'
    print(f'Lignes stables apres join credit card : {len(df)}')
    del cc
    gc.collect()

Shape credit card agrégé : (103558, 101)
Lignes stables apres join credit card : 356255
Credit card balance - fait en 35s


## 9. Dataset final

On sépare à nouveau train et test, et on vérifie le résultat.

In [22]:
train_df = df[df['TARGET'].notnull()].copy()
test_df  = df[df['TARGET'].isnull()].copy()

print('Train enrichi :', train_df.shape)
print('Test  enrichi :', test_df.shape)

del df
gc.collect()

Train enrichi : (307511, 398)
Test  enrichi : (48744, 398)


0

## 10. Export des données enrichies

In [23]:
train_df.to_csv('data/train_engineered.csv', index=False)
test_df.to_csv('data/test_engineered.csv',   index=False)

print('Données exportées :')
print(f'  data/train_engineered.csv : {train_df.shape}')
print(f'  data/test_engineered.csv  : {test_df.shape}')

Données exportées :
  data/train_engineered.csv : (307511, 398)
  data/test_engineered.csv  : (48744, 398)
